# Open-Loop vs Closed-Loop Control: Unicycle Robot

This notebook compares an open-loop input with a feedback policy for driving a unicycle robot from the origin to a goal position:

- **Open-loop:** the input sequence is computed in advance and applied without using any measurement of the robot's state.
- **Closed-loop (feedback):** at each step, the input is computed from the robot's current position and heading.

A small random steering disturbance is added to the dynamics, and the resulting trajectories of the two strategies are compared.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Unicycle model

The state is $s = (x, y, \phi)$: the position $(x, y)$ and the heading angle $\phi$. The action (input) is $a = (v, u)$: the forward velocity $v$ and the angular velocity $u$.

Using an Euler discretization with step $\delta$, and a small random steering disturbance $w_k \sim \mathcal{N}(0, \sigma^2)$:

$$
\begin{aligned}
x_{k+1} &= x_k + \delta \, v_k \cos\phi_k \\
y_{k+1} &= y_k + \delta \, v_k \sin\phi_k \\
\phi_{k+1} &= \phi_k + \delta \, u_k + w_k
\end{aligned}
$$

The robot starts at $(0, 0)$ with heading $\phi = 0$. The objective is to reach the goal $(x_\text{goal}, y_\text{goal})$.

In [ ]:
delta = 0.1   # discretization step
steps = 100   # number of simulation steps
x_goal, y_goal = 4.0, 3.0

## 2. Open-loop input design

A unicycle starting at $(0, 0)$ with heading $\phi = 0$ and constant inputs $v$ and $u$ traces a circle of radius $r = v/u$. We choose $r$ so that this circle passes through the goal:

- `u_plan = v_plan / radius` is the constant angular velocity that keeps the robot on this circle.
- `travel_time` is the time required for the arc to reach the goal; after this time, the inputs are set to zero.

Without noise, this arc reaches the goal.

In [ ]:
radius = (x_goal**2 + y_goal**2) / (2*y_goal)
turn_angle = 2*np.arctan2(y_goal, x_goal)
v_plan = 1.0                      # open-loop forward velocity
u_plan = v_plan / radius          # open-loop angular velocity
travel_time = radius * turn_angle / v_plan

## 3. Closed-loop policy and simulation

The feedback policy uses the current position and heading to steer the robot toward the goal:

- the desired heading $\phi_\text{des}$ is the direction from the current position to the goal, and the heading error $e = \phi_\text{des} - \phi$ is normalized to $[-\pi, \pi]$;
- the angular velocity is proportional to the heading error: $u = 2e$;
- the forward velocity is reduced near the goal to avoid overshoot: $v = \min(1, d)$, where $d$ is the distance to the goal.

The function `simulate` below simulates either strategy for a given noise sequence.

In [ ]:
N_RUNS = 20       # number of stochastic trials
NOISE_STD = 0.015 # steering noise standard deviation (radians)
rng = np.random.default_rng(7)

In [ ]:
def simulate(closed_loop, noise):
    x, y, phi = 0.0, 0.0, 0.0  # initial condition
    xs, ys = [x], [y]

    for k in range(steps):
        if closed_loop:
            # Feedback: use current position and heading to steer toward goal.
            distance = np.sqrt((x_goal - x)**2 + (y_goal - y)**2)  # distance to goal
            desired_phi = np.arctan2(y_goal-y, x_goal-x)           # heading that points toward goal
            error = (desired_phi-phi+np.pi) % (2*np.pi) - np.pi    # heading error, normalized to [-pi, pi]
            v = min(1.0, distance)                                  # reduce speed near goal to avoid overshoot
            u = 2.0 * error                                         # proportional angle control
        else:
            # Open-loop: planned input sequence computed in advance.
            remaining = max(0.0, travel_time-k*delta)
            v = v_plan * min(1.0, remaining/delta)
            u = u_plan if remaining > 0 else 0.0

        # Euler discretization with a small random steering disturbance.
        x_next = x + delta*v*np.cos(phi)
        y_next = y + delta*v*np.sin(phi)
        phi_next = phi + delta*u + noise[k]
        x, y, phi = x_next, y_next, phi_next
        xs.append(x)
        ys.append(y)

    return np.array(xs), np.array(ys)

## 4. Single run

A single noise realization is drawn and applied to both strategies, so that both are subjected to the same disturbance.

In [ ]:
noise_single = rng.normal(0, NOISE_STD, steps)
x_open,   y_open   = simulate(False, noise_single)
x_closed, y_closed = simulate(True,  noise_single)

plt.figure(figsize=(7, 5))
plt.plot(x_open,   y_open,   label="Open-loop (planned arc)", linewidth=2)
plt.plot(x_closed, y_closed, label="Closed-loop (feedback)",  linewidth=2)
plt.scatter([0],      [0],      c="black", marker="s",  label="Start")
plt.scatter([x_goal], [y_goal], c="red",   marker="*", s=170, label="Goal")
plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title("Same small steering disturbance for both robots")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Open-loop distance to goal:   {np.sqrt((x_open[-1]-x_goal)**2  + (y_open[-1]-y_goal)**2):.2f} m")
print(f"Closed-loop distance to goal: {np.sqrt((x_closed[-1]-x_goal)**2 + (y_closed[-1]-y_goal)**2):.2f} m")

## 5. Multiple runs

A single noise realization is not necessarily representative. The simulation is therefore repeated `N_RUNS` times for each strategy, with an independent noise sequence drawn for each run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
titles = ["Open-loop (planned arc)", "Closed-loop (feedback)"]

for ax, closed_loop, title in zip(axes, [False, True], titles):
    for i in range(N_RUNS):
        noise = rng.normal(0, NOISE_STD, steps)   # fresh noise each run
        xs, ys = simulate(closed_loop, noise)
        color = "tomato" if not closed_loop else "steelblue"
        ax.plot(xs, ys, color=color, alpha=0.35, linewidth=1)

    ax.scatter([0],      [0],      c="black", marker="s",  s=60,  zorder=5, label="Start")
    ax.scatter([x_goal], [y_goal], c="red",   marker="*", s=200, zorder=5, label="Goal")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel("x (m)")
    ax.axis("equal")
    ax.grid(True, alpha=0.4)
    ax.legend()

axes[0].set_ylabel("y (m)")
fig.suptitle(f"{N_RUNS} stochastic runs, noise std = {NOISE_STD} rad", fontsize=13)
plt.tight_layout()
plt.show()